# V5: remove bottle, improve the unified detector first

Use **Runtime → Change runtime type → GPU**. Start with the 40-epoch baseline. Download its report before committing to the full campaign.

Six classes: food, plate, utensil, hinged_container, drink_carton, wrapper. Annotations are **provisional**. This notebook contains the Python code; it needs no GitHub branch download. No training has been performed to produce this handoff.

Required inputs: the corrected complete `YOLO_V5_CommandCenter_B187_QC_selected_merged1572.zip` and the documented 28-image ambiguity exclusion ledger. The old audit320 ZIP is not substituted. Specialists are deferred until actual unified results and repeat/uncertainty analysis justify them.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics==8.4.175', 'PyYAML>=6.0', 'Pillow>=10.0', 'ImageHash==4.3.2'], check=True)
import torch
assert torch.cuda.is_available(), 'Enable a GPU runtime before training.'
print('GPU:', torch.cuda.get_device_name(0))


## 1. Save data and runs in a persistent folder

The default mounts your Google Drive so runtime resets do not erase checkpoints. Keep the same WORK path when reopening this notebook. If you turn persistence off, download results before ending the session.


In [ ]:
from pathlib import Path
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/cafeteria-v5-six-class')
else:
    WORK = Path('/content/cafeteria-v5-six-class')
WORK.mkdir(parents=True, exist_ok=True)
CODE = Path('/content/v5-six-class-code')
CODE.mkdir(exist_ok=True)
DATA = WORK / 'data'
RUNS = WORK / 'runs'
print('Persistent work directory:', WORK)


## 2. Install the included experiment scripts

This collapsed cell contains the source snapshot for the CLI, dataset preparation and trial runner. It does not train anything.


In [ ]:
SOURCES = {'v5_dataset.py': '"""Audited, isolated six-class views of the corrected V5 archive."""\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nimport io\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport stat\nimport zipfile\n\nfrom PIL import Image, ImageOps\nimport yaml\n\nNAMES = [\'food\', \'plate\', \'utensil\', \'hinged_container\', \'drink_carton\', \'wrapper\']\nARCHIVE_NAME = \'YOLO_V5_CommandCenter_B187_QC_selected_merged1572.zip\'\nAUDIT320_NAME = \'YOLO_V5_partial_annotation_audit320(5).zip\'\nPREFIX = \'v5_original_plus_v4/\'\nSPLITS = (\'train\', \'val\', \'test\')\n\n\ndef digest(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef write_json(path, value):\n    with Path(path).open(\'x\', encoding=\'utf-8\') as f:\n        json.dump(value, f, indent=2, allow_nan=False)\n        f.write(\'\\n\')\n\n\ndef safe_path(name):\n    p = PurePosixPath(name)\n    if not name or \'\\\\\' in name or p.is_absolute() or \'..\' in p.parts or str(p) != name:\n        raise ValueError(f\'Unsafe or noncanonical path: {name}\')\n    return name\n\n\ndef strip_bottle(text):\n    kept, removed = [], 0\n    for line in text.splitlines():\n        if not line.strip():\n            continue\n        p = line.split()\n        if len(p) != 5 or p[0] not in {str(i) for i in range(7)}:\n            raise ValueError(\'Invalid YOLO class or row\')\n        x, y, w, h = map(float, p[1:])\n        if not (all(math.isfinite(v) for v in (x, y, w, h)) and 0 < w <= 1 and 0 < h <= 1\n                and -1e-5 <= x-w/2 <= x+w/2 <= 1+1e-5\n                and -1e-5 <= y-h/2 <= y+h/2 <= 1+1e-5):\n            raise ValueError(\'Invalid YOLO geometry\')\n        if p[0] == \'6\':\n            removed += 1\n        else:\n            kept.append(\' \'.join(p))\n    return \'\\n\'.join(kept) + (\'\\n\' if kept else \'\'), removed\n\n\ndef read_archive(path, expected_images=2261):\n    path = Path(path).expanduser().resolve()\n    if not path.is_file():\n        raise FileNotFoundError(f\'Corrected V5 archive is unavailable: {path}\')\n    contents, removed, counts = {}, Counter(), {s: Counter() for s in SPLITS}\n    with zipfile.ZipFile(path) as z:\n        names = set()\n        for entry in z.infolist():\n            name = entry.filename.rstrip(\'/\') if entry.is_dir() else entry.filename\n            safe_path(name)\n            if not name.startswith(PREFIX) and name != PREFIX.rstrip(\'/\'):\n                raise ValueError(f\'Unexpected archive path: {name}\')\n            if entry.filename in names:\n                raise ValueError(f\'Duplicate ZIP member: {entry.filename}\')\n            names.add(entry.filename)\n            if stat.S_ISLNK(entry.external_attr >> 16):\n                raise ValueError(\'Archive symlink is not allowed\')\n        rows = json.loads(z.read(PREFIX + \'manifest.json\'))\n        if not isinstance(rows, list) or len(rows) != expected_images:\n            raise ValueError(f\'Expected {expected_images} manifest images\')\n        cfg = yaml.safe_load(z.read(PREFIX + \'data.yaml\'))\n        names_cfg = cfg[\'names\']\n        if isinstance(names_cfg, dict):\n            names_cfg = [names_cfg.get(i) for i in range(7)]\n        if names_cfg != NAMES + [\'bottle\']:\n            raise ValueError(\'Wrong original seven-class mapping\')\n        split_counts = Counter(r[\'split\'] for r in rows)\n        if set(split_counts) != set(SPLITS):\n            raise ValueError(\'Expected train/val/test splits\')\n        if expected_images == 2261 and split_counts != dict(train=2093, val=97, test=71):\n            raise ValueError(\'Unexpected original V5 split counts\')\n        seen_paths = set()\n        for r in rows:\n            for kind in (\'image\', \'label\'):\n                rel = safe_path(r[kind + \'_path\'])\n                expected_prefix = (\'images/\' if kind == \'image\' else \'labels/\') + r[\'split\'] + \'/\'\n                if not rel.startswith(expected_prefix):\n                    raise ValueError(\'Manifest path and split disagree\')\n                if rel in seen_paths:\n                    raise ValueError(\'Duplicate manifest path\')\n                seen_paths.add(rel)\n                data = z.read(PREFIX + rel)\n                if digest(data) != r[kind + \'_sha256\']:\n                    raise ValueError(f\'Source checksum mismatch: {rel}\')\n                contents[rel] = data\n            ip, lp = PurePosixPath(r[\'image_path\']), PurePosixPath(r[\'label_path\'])\n            if ip.suffix.lower() not in {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\'} or lp.suffix != \'.txt\' or ip.stem != lp.stem:\n                raise ValueError(\'Image/label pair has incompatible names or format\')\n            r[\'label_text\'], n = strip_bottle(contents[r[\'label_path\']].decode(\'utf-8-sig\'))\n            r[\'counts\'] = dict(Counter(int(line.split()[0]) for line in r[\'label_text\'].splitlines()))\n            removed[r[\'split\']] += n\n            counts[r[\'split\']].update(r[\'counts\'])\n        # Verify CRC of auxiliary QC ledgers as well as manifest data.\n        if z.testzip() is not None:\n            raise ValueError(\'Archive CRC failure\')\n    source = dict(archive=str(path), archive_sha256=sha(path), images=len(rows),\n                  original_splits=dict(split_counts), removed_bottle_boxes=dict(removed),\n                  retained_class_counts={s: {NAMES[c]: counts[s][c] for c in range(6)} for s in SPLITS},\n                  annotation_status=\'provisional; not independently human verified\')\n    return rows, contents, source\n\n\ndef validate_exclusions(exclusions, rows):\n    paths = {r[\'image_path\'] for r in rows}\n    if len(exclusions) != 28 or len(set(exclusions)) != 28 or not set(exclusions) <= paths:\n        raise ValueError(\'Exclusions must list exactly 28 unique, exact manifest image_path values\')\n    return set(exclusions)\n\n\ndef load_exclusions(path):\n    path = Path(path)\n    if path.suffix.lower() == \'.json\':\n        items = json.loads(path.read_text())\n        if not isinstance(items, list) or any(not isinstance(p, str) for p in items):\n            raise ValueError(\'Exclusion JSON must be a list of image_path strings\')\n        return items\n    with path.open(newline=\'\', encoding=\'utf-8-sig\') as f:\n        reader = csv.DictReader(f)\n        if not reader.fieldnames or \'image_path\' not in reader.fieldnames:\n            raise ValueError(\'Exclusion CSV needs an image_path column\')\n        return [r[\'image_path\'] for r in reader]\n\n\ndef image_features(rows, contents):\n    import imagehash\n    result = []\n    for r in rows:\n        with Image.open(io.BytesIO(contents[r[\'image_path\']])) as image:\n            im = ImageOps.exif_transpose(image).convert(\'RGB\')\n            result.append(dict(phash=int(str(imagehash.phash(im)), 16),\n                               dhash=int(str(imagehash.dhash(im)), 16),\n                               aspect=im.width / im.height,\n                               pixels=digest(str(im.size).encode() + im.tobytes())))\n    return result\n\n\ndef assign_splits(rows, features):\n    n = len(rows)\n    if n < 20 or len(features) != n:\n        raise ValueError(\'Insufficient cohort for supported group-aware split\')\n    parent = list(range(n))\n\n    def root(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]\n            i = parent[i]\n        return i\n\n    def union(a, b):\n        a, b = root(a), root(b)\n        parent[max(a, b)] = min(a, b)\n\n    seen = {}\n    for i, (r, f) in enumerate(zip(rows, features)):\n        keys = [(\'bytes\', r[\'image_sha256\'])]\n        if f.get(\'pixels\'):\n            keys.append((\'pixels\', f[\'pixels\']))\n        for key in (\'meal_id\', \'tray_pair_id\', \'pair_id\', \'group_id\', \'session_id\'):\n            if r.get(key) is not None and str(r[key]).strip():\n                keys.append((key, r.get(\'dataset\', r.get(\'source_dataset\', \'\')), str(r[key])))\n        for key in keys:\n            if key in seen:\n                union(i, seen[key])\n            else:\n                seen[key] = i\n    for i in range(n):\n        for j in range(i):\n            a, b = features[i], features[j]\n            if (abs(a[\'aspect\']-b[\'aspect\']) / max(a[\'aspect\'], b[\'aspect\']) <= .025\n                    and (a[\'phash\'] ^ b[\'phash\']).bit_count() <= 3\n                    and (a[\'dhash\'] ^ b[\'dhash\']).bit_count() <= 4):\n                union(i, j)\n    grouped = defaultdict(list)\n    for i in range(n):\n        grouped[root(i)].append(i)\n    groups = {}\n    selected = {}\n    size = Counter()\n    boxes = {s: Counter() for s in SPLITS}\n    for gid, inds in grouped.items():\n        counts = sum((Counter(rows[i][\'counts\']) for i in inds), Counter())\n        groups[gid] = dict(members=inds, size=len(inds), counts=counts)\n        held = {rows[i][\'split\'] for i in inds}\n        fixed = \'test\' if \'test\' in held else \'val\' if \'val\' in held else None\n        if fixed:\n            selected[gid] = fixed\n            size[fixed] += len(inds)\n            boxes[fixed].update(counts)\n    target = round(n * .1)\n    if any(size[s] > .12 * n for s in (\'val\', \'test\')):\n        raise ValueError(\'Held-out duplicate groups prevent safe 80/10/10 split\')\n    remaining = set(groups) - set(selected)\n\n    def take(gid, split):\n        selected[gid] = split\n        remaining.remove(gid)\n        size[split] += groups[gid][\'size\']\n        boxes[split].update(groups[gid][\'counts\'])\n\n    frequency = {c: sum(g[\'counts\'][c] > 0 for g in groups.values()) for c in range(6)}\n    for c in sorted(range(6), key=lambda c: (frequency[c], c)):\n        minimum = 2 if frequency[c] >= 6 else 1\n        if frequency[c] < 3:\n            raise ValueError(f\'Insufficient independent group support: {NAMES[c]}\')\n        for s in (\'val\', \'test\'):\n            while sum(groups[g][\'counts\'][c] > 0 for g in selected if selected[g] == s) < minimum:\n                options = [g for g in remaining if groups[g][\'counts\'][c] > 0\n                           and size[s] + groups[g][\'size\'] <= target]\n                if not options:\n                    raise ValueError(f\'Cannot allocate independent class support for {NAMES[c]} in {s}\')\n                take(min(options, key=lambda g: (groups[g][\'size\'], -min(groups[g][\'counts\'][c], 5), g)), s)\n    total = sum((g[\'counts\'] for g in groups.values()), Counter())\n    for s in (\'val\', \'test\'):\n        def score(g):\n            return (sum(((boxes[s][c]+groups[g][\'counts\'][c]-total[c]*.1)**2\n                         - (boxes[s][c]-total[c]*.1)**2) / max(1, total[c]*.1)\n                        for c in range(6)) / groups[g][\'size\'], groups[g][\'size\'], g)\n        while size[s] < target:\n            options = [g for g in remaining if groups[g][\'size\'] <= target-size[s]]\n            if not options:\n                break\n            take(min(options, key=score), s)\n    for g in remaining:\n        selected[g] = \'train\'\n        size[\'train\'] += groups[g][\'size\']\n        boxes[\'train\'].update(groups[g][\'counts\'])\n    if any(not .08*n <= size[s] <= .12*n for s in (\'val\', \'test\')):\n        raise ValueError(\'Group sizes prevent safe 80/10/10 split\')\n    if any(boxes[s][c] == 0 for s in SPLITS for c in range(6)):\n        raise ValueError(\'Missing class support after splitting\')\n    return [selected[root(i)] for i in range(n)], [root(i) for i in range(n)]\n\n\ndef materialize(rows, contents, source, splits, groups, output, exclusions):\n    output = Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(f\'Refusing to overwrite dataset: {output}\')\n    output.mkdir(parents=True)\n    records, files = [], {}\n    counts = {s: Counter() for s in SPLITS}\n    for i, (row, split, group) in enumerate(zip(rows, splits, groups)):\n        stem = f\'{i:05d}\'\n        ip = f\'images/{split}/{stem}{PurePosixPath(row["image_path"]).suffix.lower()}\'\n        lp = f\'labels/{split}/{stem}.txt\'\n        for rel, data in ((ip, contents[row[\'image_path\']]), (lp, row[\'label_text\'].encode())):\n            p = output / rel\n            p.parent.mkdir(parents=True, exist_ok=True)\n            p.write_bytes(data)\n            files[rel] = digest(data)\n        records.append(dict(image_path=ip, label_path=lp, image_sha256=files[ip],\n                            label_sha256=files[lp], split=split, group=group,\n                            original_split=row[\'split\'], original_image=row[\'image_path\'],\n                            source_label_sha256=row[\'label_sha256\'], counts=row[\'counts\']))\n        counts[split].update(row[\'counts\'])\n    for split in SPLITS:\n        rel = split + \'.txt\'\n        (output / rel).write_text(\'\'.join(str(output / r[\'image_path\']) + \'\\n\'\n                                        for r in records if r[\'split\'] == split))\n        files[rel] = sha(output / rel)\n    (output / \'data.yaml\').write_text(yaml.safe_dump(dict(\n        path=str(output), train=\'train.txt\', val=\'val.txt\', test=\'test.txt\', nc=6, names=NAMES), sort_keys=False))\n    files[\'data.yaml\'] = sha(output / \'data.yaml\')\n    receipt = dict(schema=1, names=NAMES, source=source, files=files, images=records,\n                   exclusions=sorted(exclusions), split_counts=dict(Counter(splits)),\n                   class_counts={s: {NAMES[c]: counts[s][c] for c in range(6)} for s in SPLITS},\n                   annotation_status=\'provisional; not independently human verified\')\n    write_json(output / \'dataset_receipt.json\', receipt)\n    return receipt\n\n\ndef source_policy(name, allow_provisional_audit320=False, exclusions=None):\n    if name == AUDIT320_NAME:\n        if not allow_provisional_audit320:\n            raise ValueError(\'Older audit320 data requires explicit source authorization\')\n        return dict(dataset_revision=\'audit320\', source_choice=\'user-authorized older provisional archive\',\n                    later_corrections_missing=True, later_28_image_exclusions_applied=False,\n                    limitation=\'Later B187 corrections and its 28-image exclusion ledger are unavailable; \'\n                               \'their identities are not guessed or fabricated for this older revision.\')\n    if name != ARCHIVE_NAME:\n        raise ValueError(f\'Use {ARCHIVE_NAME}, or explicitly authorize the known local audit320 archive\')\n    if exclusions is None:\n        raise ValueError(\'The corrected B187 archive requires its 28-image exclusion ledger\')\n    return dict(dataset_revision=\'B187_selected\', later_corrections_missing=False,\n                later_28_image_exclusions_applied=True)\n\n\ndef prepare(archive, output, exclusions=None, allow_provisional_audit320=False):\n    # Check existence first to preserve the useful missing-input error.\n    if not Path(archive).is_file():\n        raise FileNotFoundError(f\'V5 archive is unavailable: {archive}\')\n    policy = source_policy(Path(archive).name, allow_provisional_audit320, exclusions)\n    if Path(output).exists():\n        raise FileExistsError(output)\n    rows, contents, source = read_archive(archive)\n    source.update(policy)\n    excluded = validate_exclusions(load_exclusions(exclusions), rows) if exclusions else set()\n    source[\'exclusion_ledger_sha256\'] = sha(exclusions) if exclusions else None\n    if policy[\'dataset_revision\'] == \'audit320\':\n        source[\'later_28_image_exclusions_applied\'] = False\n    rows = [r for r in rows if r[\'image_path\'] not in excluded]\n    features = image_features(rows, contents)\n    splits, groups = assign_splits(rows, features)\n    return materialize(rows, contents, source, splits, groups, output, excluded)\n\n\ndef verify_dataset(output, require_support=True):\n    output = Path(output).resolve()\n    receipt = json.loads((output / \'dataset_receipt.json\').read_text())\n    if receipt[\'names\'] != NAMES:\n        raise ValueError(\'Changed six-class names\')\n    for rel, expected in receipt[\'files\'].items():\n        safe_path(rel)\n        p = output / rel\n        if not p.is_file() or p.is_symlink() or sha(p) != expected:\n            raise ValueError(f\'Changed dataset file: {rel}\')\n    group_splits, image_splits, seen = {}, {}, set()\n    actual = {s: Counter() for s in SPLITS}\n    for r in receipt[\'images\']:\n        s = r[\'split\']\n        if s not in SPLITS or r[\'image_path\'] in seen:\n            raise ValueError(\'Duplicate image or invalid split\')\n        seen.add(r[\'image_path\'])\n        for kind in (\'image\', \'label\'):\n            rel = safe_path(r[kind + \'_path\'])\n            if receipt[\'files\'].get(rel) != r[kind + \'_sha256\']:\n                raise ValueError(\'Receipt image/label hashes disagree\')\n        text, removed = strip_bottle((output / r[\'label_path\']).read_text())\n        if removed:\n            raise ValueError(\'Bottle remains in six-class dataset\')\n        counts = Counter(int(line.split()[0]) for line in text.splitlines())\n        if dict(counts) != {int(c): n for c, n in r[\'counts\'].items()}:\n            raise ValueError(\'Receipt class counts disagree\')\n        actual[s].update(counts)\n        if r[\'original_split\'] != \'train\' and s == \'train\':\n            raise ValueError(\'Historical held-out image entered training\')\n        for mapping, key in ((group_splits, r[\'group\']), (image_splits, r[\'image_sha256\'])):\n            if key in mapping and mapping[key] != s:\n                raise ValueError(\'Group/image overlap between splits\')\n            mapping[key] = s\n    cfg = yaml.safe_load((output / \'data.yaml\').read_text())\n    if cfg != dict(path=str(output), train=\'train.txt\', val=\'val.txt\', test=\'test.txt\', nc=6, names=NAMES):\n        raise ValueError(\'Changed dataset configuration; dataset cannot be moved without rebuilding lists\')\n    for s in SPLITS:\n        expected = [str(output / r[\'image_path\']) for r in receipt[\'images\'] if r[\'split\'] == s]\n        if (output / (s + \'.txt\')).read_text().splitlines() != expected:\n            raise ValueError(\'Image list differs from receipt\')\n        if require_support and any(actual[s][c] == 0 for c in range(6)):\n            raise ValueError(f\'Missing class support in {s}\')\n        if receipt[\'class_counts\'][s] != {NAMES[c]: actual[s][c] for c in range(6)}:\n            raise ValueError(\'Changed class support totals\')\n    return receipt\n', 'v5_trials.py': '"""Unified V5 trials. Specialist training is deliberately unavailable at this stage."""\nfrom collections import Counter\nimport csv\nimport json\nimport math\nfrom pathlib import Path\nimport platform\nimport shutil\nimport time\n\nimport yaml\n\nfrom v5_dataset import NAMES, sha, safe_path, verify_dataset, write_json\n\nTRIALS = (\'baseline\', \'U1\', \'U2\', \'U3\', \'U4\', \'U5\', \'U6\')\nSEEDS = (57, 58, 59)\n\n\ndef ensure_disk_space(output, minimum_bytes=3 * 1024**3):\n    ancestor = Path(output).resolve()\n    while not ancestor.exists():\n        ancestor = ancestor.parent\n    free = shutil.disk_usage(ancestor).free\n    if free < minimum_bytes:\n        raise ValueError(f\'Insufficient disk space: {free / 1024**3:.2f} GiB free; \'\n                         f\'at least {minimum_bytes / 1024**3:.1f} GiB required for data/checkpoints/swap\')\n\n\ndef operating_point(confidence, precision, recall):\n    # Zero predictions do not establish an attainable precision target.\n    options = [(float(r), float(x)) for x, p, r in zip(confidence, precision, recall)\n               if p >= .8 and r > 0]\n    if not options:\n        return dict(recall_at_precision_80=0.0, threshold_at_precision_80=None)\n    r, x = max(options, key=lambda pair: (pair[0], -pair[1]))\n    return dict(recall_at_precision_80=r, threshold_at_precision_80=x)\n\n\ndef weak_classes(record):\n    return [name for name in NAMES if record[\'per_class\'][name][\'AP50\'] < .8\n            or record[\'per_class\'][name][\'recall_at_precision_80\'] < .7]\n\n\ndef valid_completed(record, dataset_sha):\n    if (record.get(\'status\') != \'complete\' or record.get(\'smoke\') is not False\n            or record.get(\'dataset_sha256\') != dataset_sha\n            or record.get(\'checkpoint_names\') != NAMES\n            or record.get(\'test_evaluated\') is not False\n            or record.get(\'trial\') not in TRIALS or record.get(\'seed\') not in SEEDS):\n        return False\n    if (type(record.get(\'completed_epochs\')) is not int or type(record.get(\'planned_epochs\')) is not int\n            or record[\'completed_epochs\'] < record[\'planned_epochs\']\n            or record[\'planned_epochs\'] < (80 if record.get(\'recipe\', {}).get(\'epochs\') == 80 else 40)):\n        return False\n    per_class = record.get(\'per_class\', {})\n    if set(per_class) != set(NAMES):\n        return False\n    for values in per_class.values():\n        for k in (\'AP50\', \'AP50_95\', \'recall_at_precision_80\'):\n            value = values.get(k)\n            if type(value) not in (float, int) or not math.isfinite(value) or not 0 <= value <= 1:\n                return False\n        threshold = values.get(\'threshold_at_precision_80\')\n        if threshold is not None and (type(threshold) not in (float, int)\n                                      or not math.isfinite(threshold) or not 0 <= threshold <= 1):\n            return False\n        if values[\'recall_at_precision_80\'] > 0 and threshold is None:\n            return False\n    return True\n\n\ndef select_candidate(runs):\n    by_trial = {r[\'trial\']: r for r in runs if r[\'seed\'] == 57}\n    baseline = by_trial[\'baseline\']\n    weak = weak_classes(baseline)\n    acceptable = [r for r in by_trial.values() if all(\n        r[\'per_class\'][n][\'AP50_95\'] >= baseline[\'per_class\'][n][\'AP50_95\'] - .02 - 1e-12\n        for n in NAMES)]\n    # Preserve baseline on ties or when it has no weak classes.\n    if not weak:\n        return baseline\n    return max(acceptable, key=lambda r: (\n        sum(r[\'per_class\'][n][\'AP50_95\'] for n in weak) / len(weak),\n        r[\'trial\'] == \'baseline\', -TRIALS.index(r[\'trial\'])))\n\n\ndef trial_recipe(trial, seed, completed=()):\n    if trial not in TRIALS or seed not in SEEDS:\n        raise ValueError(\'Unknown trial or seed; use baseline/U1-U6 and seeds 57/58/59\')\n    recipe = dict(pretrained=\'yolo11s.pt\', epochs=40, patience=40, imgsz=800, batch=4,\n                  optimizer=\'AdamW\', lr0=.0005, lrf=.01, weight_decay=.0005,\n                  cos_lr=True, seed=seed, deterministic=True, workers=0,\n                  amp=True, cache=False, nbs=64, degrees=7, translate=.1, scale=.25,\n                  fliplr=.5, flipud=0, mosaic=.5, mixup=0, close_mosaic=5,\n                  plots=True, save=True, save_period=-1, balance=False)\n    changes = dict(U1=dict(scale=.1, mosaic=.1, close_mosaic=10),\n                   U2=dict(imgsz=1024, batch=2), U3=dict(balance=True),\n                   U4=dict(lr0=.0002), U5=dict(pretrained=\'yolo11m.pt\', batch=2))\n    if trial == \'U6\':\n        by = {r[\'trial\']: r for r in completed if r.get(\'seed\') == 57}\n        if not all(k in by for k in TRIALS[:-1]):\n            raise ValueError(\'U6 requires completed baseline and U1 through U5 prerequisites\')\n        baseline = by[\'baseline\']\n        weak = weak_classes(baseline)\n        supported = [r for k, r in by.items() if k in changes and\n                     all(r[\'per_class\'][n][\'AP50_95\'] >= baseline[\'per_class\'][n][\'AP50_95\'] - .02 - 1e-12\n                         for n in NAMES) and any(\n                             r[\'per_class\'][n][\'AP50_95\'] >= baseline[\'per_class\'][n][\'AP50_95\'] + .02\n                             or r[\'per_class\'][n][\'recall_at_precision_80\'] >=\n                             baseline[\'per_class\'][n][\'recall_at_precision_80\'] + .05 for n in weak)]\n        if supported:\n            for r in sorted(supported, key=lambda r: TRIALS.index(r[\'trial\'])):\n                recipe.update(changes[r[\'trial\']])\n        else:\n            recipe.update(epochs=80, patience=80)\n    elif trial in changes:\n        recipe.update(changes[trial])\n    return recipe\n\n\ndef with_epoch_budget(recipe, epochs):\n    if epochs is None:\n        return dict(recipe)\n    if type(epochs) is not int or epochs < 1:\n        raise ValueError(\'Epoch budget must be a positive integer\')\n    return dict(recipe, epochs=epochs, patience=epochs,\n                close_mosaic=min(recipe[\'close_mosaic\'], epochs - 1))\n\n\ndef balanced_training_paths(records, weak):\n    weak_ids = {NAMES.index(n) for n in weak}\n    paths = []\n    for r in records:\n        if r[\'split\'] != \'train\':\n            continue\n        positive = any(int(c) in weak_ids and count > 0 for c, count in r[\'counts\'].items())\n        paths.extend([r[\'image_path\']] * (3 if positive else 1))\n    return paths\n\n\ndef verify_artifacts(root, receipt):\n    root = Path(root).resolve()\n    for rel, expected in receipt.get(\'artifacts\', {}).items():\n        safe_path(rel)\n        path = root / rel\n        if not path.is_file() or path.is_symlink() or sha(path) != expected:\n            raise ValueError(f\'Changed run artifact: {rel}\')\n\n\ndef load_runs(root, dataset_sha):\n    records = []\n    root = Path(root)\n    if not root.exists():\n        return records\n    for path in sorted(root.glob(\'*/run.json\')):\n        record = json.loads(path.read_text())\n        if record.get(\'dataset_sha256\') != dataset_sha:\n            continue\n        if record.get(\'status\') == \'complete\':\n            required = {\'fit/weights/best.pt\', \'fit/results.csv\', \'configuration.json\', \'train.txt\',\n                        \'run_data.yaml\', \'validation_stats.json\', \'curves.json\'}\n            if not required <= record.get(\'artifacts\', {}).keys():\n                raise ValueError(f\'Run is missing required artifact provenance: {path}\')\n            verify_artifacts(path.parent, record)\n            cfg = json.loads((path.parent / \'configuration.json\').read_text())\n            if any(record.get(k) != cfg.get(k) for k in (\'trial\', \'seed\', \'smoke\', \'dataset_sha256\', \'recipe\')):\n                raise ValueError(f\'Run configuration differs from receipt: {path}\')\n            if record.get(\'planned_epochs\') != cfg[\'recipe\'][\'epochs\']:\n                raise ValueError(\'Run epoch budget differs from configuration\')\n            with (path.parent / \'fit/results.csv\').open() as f:\n                actual_epochs = len(list(csv.DictReader(f)))\n            if actual_epochs != record.get(\'completed_epochs\'):\n                raise ValueError(\'Training log epoch count differs from receipt\')\n        records.append(record)\n    return records\n\n\ndef summarize_stage(runs, dataset_sha):\n    completed = [r for r in runs if valid_completed(r, dataset_sha)]\n    keys = [(r[\'trial\'], r[\'seed\']) for r in completed]\n    if len(keys) != len(set(keys)):\n        raise ValueError(\'Duplicate completed trial/seed; use a separate experiment root\')\n    missing = [k for k in TRIALS if (k, 57) not in keys]\n    report = dict(specialists_allowed=False, missing_trials=missing,\n                  completed_runs=[dict(trial=k, seed=s) for k, s in keys],\n                  status=\'unified_trials_required\', test_evaluated=False)\n    if missing:\n        return report\n    selected = select_candidate(completed)\n    report[\'selected_unified_trial\'] = selected[\'trial\']\n    report[\'baseline_weak_classes\'] = weak_classes(next(r for r in completed if r[\'trial\'] == \'baseline\' and r[\'seed\'] == 57))\n    needed = [(k, seed) for k in dict.fromkeys((\'baseline\', selected[\'trial\'])) for seed in (58, 59)\n              if (k, seed) not in keys]\n    report[\'required_confirmation_runs\'] = [dict(trial=k, seed=s) for k, s in needed]\n    report[\'status\'] = \'confirmation_runs_required\' if needed else \'uncertainty_analysis_required\'\n    report[\'note\'] = (\'No specialist eligibility has been established. After repeats, inspect group support, \'\n                      \'paired group-bootstrap intervals, seed variability and remaining class deficits \'\n                      \'before implementing a specialist. This status cannot be overridden by a CLI flag.\')\n    return report\n\n\ndef stage_report(dataset, runs):\n    verify_dataset(dataset)\n    dataset_sha = sha(Path(dataset) / \'dataset_receipt.json\')\n    return summarize_stage(load_runs(runs, dataset_sha), dataset_sha)\n\n\ndef checkpoint_names(model):\n    names = model.names\n    return [names[i] for i in range(len(names))] if isinstance(names, dict) else list(names)\n\n\ndef validate_model(model):\n    if checkpoint_names(model) != NAMES:\n        raise ValueError(\'Checkpoint must have exactly six retained classes, in the fixed order\')\n\n\ndef verify_runtime(previous, current):\n    if any(previous.get(k) != current.get(k) for k in (\'python\', \'torch\', \'ultralytics\')):\n        raise ValueError(\'Cannot resume in a different Python/Torch/Ultralytics environment\')\n\n\ndef resume_action(previous, checkpoint, epochs):\n    if previous.get(\'phase\') == \'evaluation\' and previous.get(\'completed_epochs\') == epochs:\n        return \'evaluate\'\n    epoch = checkpoint.get(\'epoch\')\n    if type(epoch) is not int or not 0 <= epoch < epochs - 1 or checkpoint.get(\'optimizer\') is None:\n        raise ValueError(\'Checkpoint is not resumable; refusing to restart training\')\n    return \'train\'\n\n\ndef clear_label_caches(dataset):\n    # cache=False disables image caching only. Rebuild derived label caches from\n    # the verified text labels, including before validation/recovery loads.\n    for split in (\'train\', \'val\', \'test\'):\n        path = Path(dataset) / \'labels\' / (split + \'.cache\')\n        if path.is_file() or path.is_symlink():\n            path.unlink()\n\n\ndef evaluate(model, data_file, output, recipe, device, batch):\n    """Capture per-image matching stats before Ultralytics clears them."""\n    from ultralytics.models.yolo.detect.val import DetectionValidator\n\n    class CaptureValidator(DetectionValidator):\n        def update_metrics(self, preds, batch_data):\n            if not hasattr(self, \'v5_image_names\'):\n                self.v5_image_names = []\n            self.v5_image_names.extend(Path(p).name for p in batch_data[\'im_file\'])\n            super().update_metrics(preds, batch_data)\n\n        def get_stats(self):\n            per_image = []\n            for i, name in enumerate(self.v5_image_names):\n                per_image.append(dict(name=name, **{\n                    k: self.metrics.stats[k][i].tolist() for k in (\'tp\', \'conf\', \'pred_cls\', \'target_cls\')}))\n            write_json(output / \'validation_stats.json\', per_image)\n            return super().get_stats()\n\n    validate_model(model)\n    clear_label_caches(Path(yaml.safe_load(Path(data_file).read_text())[\'path\']))\n    metrics = model.val(validator=CaptureValidator, data=str(data_file), split=\'val\',\n                        imgsz=recipe[\'imgsz\'], batch=batch, device=device, workers=0,\n                        conf=.001, iou=.7, max_det=300, augment=False,\n                        plots=True, save_txt=True, save_conf=True,\n                        project=str(output), name=\'validation\', exist_ok=False)\n    box = metrics.box\n    if set(int(c) for c in box.ap_class_index) != set(range(6)):\n        raise ValueError(\'Validation metrics lack retained-class support\')\n    per_class = {}\n    for j, c in enumerate(box.ap_class_index):\n        per_class[NAMES[int(c)]] = dict(\n            AP50=float(box.ap50[j]), AP50_95=float(box.ap[j]),\n            precision_at_mean_best_f1=float(box.p[j]), recall_at_mean_best_f1=float(box.r[j]),\n            **operating_point(box.px, box.p_curve[j], box.r_curve[j]))\n    curves = dict(confidence=box.px.tolist(), class_indices=box.ap_class_index.tolist(),\n                  precision=box.p_curve.tolist(), recall=box.r_curve.tolist())\n    write_json(output / \'curves.json\', curves)\n    return per_class, {k: float(v) for k, v in metrics.speed.items()}\n\n\ndef run_trial(dataset, output, trial, seed=57, device=\'mps\', batch=None, smoke=False, resume=False, epochs=None):\n    dataset, output = Path(dataset).resolve(), Path(output).resolve()\n    ensure_disk_space(output)\n    data = verify_dataset(dataset)\n    dataset_sha = sha(dataset / \'dataset_receipt.json\')\n    runs = load_runs(output.parent, dataset_sha)\n    completed = [r for r in runs if valid_completed(r, dataset_sha)]\n    baseline = next((r for r in completed if r[\'trial\'] == \'baseline\' and r[\'seed\'] == 57), None)\n    if not smoke and trial != \'baseline\' and baseline is None:\n        raise ValueError(\'Complete the baseline at seed 57 before variants\')\n    if not smoke and seed != 57:\n        report = summarize_stage(completed, dataset_sha)\n        if report[\'missing_trials\'] or trial not in (\'baseline\', report[\'selected_unified_trial\']):\n            raise ValueError(\'Complete all unified variants before baseline/selected-candidate confirmations\')\n    recipe = with_epoch_budget(trial_recipe(trial, seed, completed), epochs)\n    if batch is not None:\n        if type(batch) is not int or batch < 1:\n            raise ValueError(\'Batch must be a positive integer\')\n        recipe[\'batch\'] = batch\n    if smoke:\n        recipe.update(epochs=2, patience=2, close_mosaic=0)\n    config = dict(schema=1, trial=trial, seed=seed, dataset_sha256=dataset_sha,\n                  recipe=recipe, smoke=smoke, device=device)\n    if output.exists() and not resume:\n        raise FileExistsError(f\'Refusing to overwrite trial: {output}\')\n    from ultralytics import YOLO\n    import ultralytics\n    import torch\n    runtime = dict(python=platform.python_version(), torch=torch.__version__,\n                   ultralytics=ultralytics.__version__, platform=platform.platform())\n    action = \'train\'\n    if device == \'mps\' and not torch.backends.mps.is_available():\n        raise ValueError(\'MPS unavailable; choose --device cpu or a CUDA device explicitly\')\n    if resume:\n        previous = json.loads((output / \'run.json\').read_text())\n        if json.loads((output / \'configuration.json\').read_text()) != config or previous[\'status\'] != \'failed\':\n            raise ValueError(\'Resume requires a failed run with exactly matching data/configuration\')\n        verify_runtime(previous.get(\'runtime\', {}), runtime)\n        verify_artifacts(output, {\'artifacts\': previous.get(\'input_artifact_hashes\', {})})\n        if set(previous.get(\'input_artifact_hashes\', {})) != {\'configuration.json\', \'run_data.yaml\', \'train.txt\'}:\n            raise ValueError(\'Failed run is missing input provenance for safe resume\')\n        if previous.get(\'phase\') == \'evaluation\' and previous.get(\'completed_epochs\') == recipe[\'epochs\']:\n            action = \'evaluate\'\n            required = {\'fit/weights/best.pt\', \'fit/results.csv\'}\n            if set(previous.get(\'training_artifact_hashes\', {})) != required:\n                raise ValueError(\'Missing completed-training provenance for evaluation recovery\')\n            verify_artifacts(output, {\'artifacts\': previous[\'training_artifact_hashes\']})\n            model = YOLO(str(output / \'fit/weights/best.pt\'))\n        else:\n            last = output / \'fit/weights/last.pt\'\n            if not last.is_file() or previous.get(\'last_checkpoint_sha256\') != sha(last):\n                raise ValueError(\'No verified resumable last.pt from the failed run\')\n            model = YOLO(str(last))\n            action = resume_action(previous, model.ckpt, recipe[\'epochs\'])\n            args = model.ckpt.get(\'train_args\', {})\n            if any(args.get(k) != value for k, value in\n                   dict(data=str(output / \'run_data.yaml\'), epochs=recipe[\'epochs\'], seed=seed).items()):\n                raise ValueError(\'Checkpoint training arguments disagree with recorded run\')\n        validate_model(model)\n    else:\n        output.mkdir(parents=True)\n        write_json(output / \'configuration.json\', config)\n        train_paths = balanced_training_paths(data[\'images\'], weak_classes(baseline)) if recipe[\'balance\'] and baseline else [\n            r[\'image_path\'] for r in data[\'images\'] if r[\'split\'] == \'train\']\n        (output / \'train.txt\').write_text(\'\'.join(str(dataset / p) + \'\\n\' for p in train_paths))\n        (output / \'run_data.yaml\').write_text(yaml.safe_dump(dict(\n            path=str(dataset), train=str(output / \'train.txt\'), val=str(dataset / \'val.txt\'), nc=6, names=NAMES), sort_keys=False))\n        model = YOLO(recipe[\'pretrained\'])\n    record = dict(config, status=\'running\', phase=\'training\', planned_epochs=recipe[\'epochs\'],\n                  test_evaluated=False, annotation_status=data[\'annotation_status\'],\n                  runtime=runtime)\n    record[\'input_artifact_hashes\'] = {p: sha(output / p) for p in\n                                      (\'configuration.json\', \'run_data.yaml\', \'train.txt\')}\n    pretrained = Path(getattr(model, \'ckpt_path\', \'\') or recipe[\'pretrained\'])\n    if pretrained.is_file():\n        record[\'initial_checkpoint_sha256\'] = sha(pretrained)\n    started = time.monotonic()\n    try:\n        # A live reserve is safer than relying only on the startup estimate,\n        # especially on the 8-GB Mac where swap may grow during training.\n        model.add_callback(\'on_train_batch_end\', lambda trainer: ensure_disk_space(\n            trainer.save_dir, minimum_bytes=1024**3))\n        if resume:\n            # Keep the old discoverable failure receipt until all checkpoint\n            # loading and provenance validation above have succeeded.\n            write_json(output / f\'failure-{time.time_ns()}.json\', previous)\n            (output / \'run.json\').unlink()\n        clear_label_caches(dataset)\n        train_args = {k: v for k, v in recipe.items() if k not in (\'pretrained\', \'balance\')}\n        if action == \'evaluate\':\n            record[\'completed_epochs\'] = previous[\'completed_epochs\']\n            record[\'phase\'] = \'evaluation\'\n            record[\'training_artifact_hashes\'] = previous[\'training_artifact_hashes\']\n            partials = [output / p for p in (\'validation\', \'validation_stats.json\', \'curves.json\')\n                        if (output / p).exists()]\n            if partials:\n                archive = output / f\'failed-validation-{time.time_ns()}\'\n                archive.mkdir()\n                for path in partials:\n                    path.rename(archive / path.name)\n        elif resume:\n            model.train(resume=True, device=device)\n        else:\n            model.train(data=str(output / \'run_data.yaml\'), device=device,\n                        project=str(output), name=\'fit\', exist_ok=False, **train_args)\n        if action != \'evaluate\':\n            record[\'completed_epochs\'] = model.trainer.epoch + 1\n        if record[\'completed_epochs\'] != recipe[\'epochs\']:\n            raise ValueError(\'Training stopped before completing the declared budget\')\n        best = output / \'fit/weights/best.pt\'\n        record[\'training_artifact_hashes\'] = {p: sha(output / p) for p in\n                                             (\'fit/weights/best.pt\', \'fit/results.csv\')}\n        record[\'phase\'] = \'evaluation\'\n        trained = YOLO(str(best))\n        validate_model(trained)\n        # Revalidate all input hashes after training before accepting metrics.\n        verify_dataset(dataset)\n        verify_artifacts(output, {\'artifacts\': record[\'input_artifact_hashes\']})\n        if sha(dataset / \'dataset_receipt.json\') != dataset_sha:\n            raise ValueError(\'Dataset receipt changed during training\')\n        record[\'per_class\'], record[\'validation_speed_ms\'] = evaluate(\n            trained, output / \'run_data.yaml\', output, recipe, device, recipe[\'batch\'])\n        stats = json.loads((output / \'validation_stats.json\').read_text())\n        expected_images = {Path(r[\'image_path\']).name for r in data[\'images\'] if r[\'split\'] == \'val\'}\n        if len(stats) != len(expected_images) or {r[\'name\'] for r in stats} != expected_images:\n            raise ValueError(\'Captured validation cohort differs from the fixed split\')\n        record.update(status=\'complete\', checkpoint_names=checkpoint_names(trained),\n                      checkpoint_sha256=sha(best), wall_seconds=time.monotonic()-started)\n        artifacts = [\'configuration.json\', \'run_data.yaml\', \'train.txt\', \'fit/results.csv\',\n                     \'fit/weights/best.pt\', \'validation_stats.json\', \'curves.json\']\n        record[\'artifacts\'] = {p: sha(output / p) for p in artifacts}\n        write_json(output / \'run.json\', record)\n    except BaseException as exc:\n        record.update(status=\'failed\', error=f\'{type(exc).__name__}: {exc}\',\n                      wall_seconds=time.monotonic()-started)\n        last = output / \'fit/weights/last.pt\'\n        if last.is_file():\n            record[\'last_checkpoint_sha256\'] = sha(last)\n        if not (output / \'run.json\').exists():\n            write_json(output / \'run.json\', record)\n        raise\n    return record\n', 'v5_experiment.py': '"""Command-line entry point for the isolated, six-class V5 experiment."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport sys\nimport zipfile\n\nfrom v5_dataset import ARCHIVE_NAME, NAMES, prepare, read_archive, verify_dataset\nfrom v5_trials import TRIALS, ensure_disk_space, run_trial, stage_report, trial_recipe\n\n\ndef campaign(dataset, runs, device, batch):\n    runs = Path(runs).resolve()\n    verify_dataset(dataset)\n\n    def one(trial, seed):\n        report = stage_report(dataset, runs)\n        if dict(trial=trial, seed=seed) in report[\'completed_runs\']:\n            print(f\'Already verified complete: {trial}, seed {seed}\', flush=True)\n            return\n        out = runs / f\'{trial}_seed{seed}\'\n        failed = out / \'run.json\'\n        resume = failed.is_file() and json.loads(failed.read_text()).get(\'status\') == \'failed\'\n        print(f\'Running {trial}, seed {seed}. All metrics use validation only.\', flush=True)\n        run_trial(dataset, out, trial, seed, device, batch, resume=resume)\n\n    for trial in TRIALS:\n        one(trial, 57)\n    report = stage_report(dataset, runs)\n    for item in report.get(\'required_confirmation_runs\', []):\n        one(item[\'trial\'], item[\'seed\'])\n    return stage_report(dataset, runs)\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\'command\', required=True)\n    audit = sub.add_parser(\'audit\', help=\'Read/verify a full source archive without extracting or training\')\n    audit.add_argument(\'--archive\', type=Path, default=Path.home() / \'Downloads\' / ARCHIVE_NAME)\n    prep = sub.add_parser(\'prepare\', help=\'Remove bottle and freeze a supported group-aware split\')\n    prep.add_argument(\'--archive\', type=Path, required=True)\n    prep.add_argument(\'--exclusions\', type=Path,\n                      help=\'28-image ambiguity exclusion CSV (image_path column) or JSON path list\')\n    prep.add_argument(\'--output\', type=Path, required=True)\n    prep.add_argument(\'--allow-provisional-audit320\', action=\'store_true\',\n                      help=\'Explicitly use the older audit320 ZIP without unavailable later QC changes\')\n    sub.add_parser(\'plan\', help=\'Print trial recipes without loading models or training\')\n    check = sub.add_parser(\'verify\', help=\'Verify a previously prepared six-class dataset\')\n    check.add_argument(\'--dataset\', type=Path, required=True)\n    status = sub.add_parser(\'status\', help=\'Report trial progress; specialists stay blocked pending analysis\')\n    status.add_argument(\'--dataset\', type=Path, required=True)\n    status.add_argument(\'--runs\', type=Path, required=True)\n    train = sub.add_parser(\'train\', help=\'Run one unified trial, using only train/val\')\n    train.add_argument(\'--trial\', choices=TRIALS, required=True)\n    train.add_argument(\'--seed\', type=int, choices=(57, 58, 59), default=57)\n    train.add_argument(\'--epochs\', type=int, help=\'Explicit epoch budget; short runs do not unlock specialists\')\n    train.add_argument(\'--output\', type=Path, required=True)\n    train.add_argument(\'--smoke\', action=\'store_true\', help=\'Two epochs; NEVER counts as a completed trial\')\n    train.add_argument(\'--resume\', action=\'store_true\', help=\'Resume verified interrupted training or recover evaluation\')\n    all_trials = sub.add_parser(\'campaign\', help=\'Run baseline, U1-U6, then required seed confirmations\')\n    all_trials.add_argument(\'--runs\', type=Path, required=True)\n    for cmd in (train, all_trials):\n        cmd.add_argument(\'--dataset\', type=Path, required=True)\n        cmd.add_argument(\'--device\', default=\'mps\', help=\'mps, cpu, or CUDA device such as 0\')\n        cmd.add_argument(\'--batch\', type=int, help=\'Override per-recipe batch only when required by memory\')\n    args = parser.parse_args(argv)\n    try:\n        if args.command == \'audit\':\n            _, _, result = read_archive(args.archive)\n        elif args.command == \'prepare\':\n            if not args.archive.is_file():\n                raise FileNotFoundError(f\'V5 archive is unavailable: {args.archive}\')\n            ensure_disk_space(args.output)\n            result = prepare(args.archive, args.output, args.exclusions, args.allow_provisional_audit320)\n        elif args.command == \'plan\':\n            result = dict(names=NAMES, order=list(TRIALS), specialists_allowed=False,\n                          recipes={k: trial_recipe(k, 57) for k in TRIALS[:-1]},\n                          U6=\'Combine supported earlier changes, or extend baseline to 80 epochs.\',\n                          confirmation_seeds=[58, 59], annotation_status=\'provisional\')\n        elif args.command == \'verify\':\n            receipt = verify_dataset(args.dataset)\n            result = dict(names=receipt[\'names\'], split_counts=receipt[\'split_counts\'],\n                          class_counts=receipt[\'class_counts\'], annotation_status=receipt[\'annotation_status\'])\n        elif args.command == \'status\':\n            result = stage_report(args.dataset, args.runs)\n        elif args.command == \'campaign\':\n            result = campaign(args.dataset, args.runs, args.device, args.batch)\n        else:\n            result = run_trial(args.dataset, args.output, args.trial, args.seed,\n                               args.device, args.batch, args.smoke, args.resume, epochs=args.epochs)\n        print(json.dumps(result, indent=2, allow_nan=False))\n        return 0\n    except (OSError, ValueError, KeyError, zipfile.BadZipFile) as exc:\n        print(f\'V5 experiment stopped: {exc}\', file=sys.stderr)\n        return 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'}
for name, source in SOURCES.items():
    path = CODE / name
    if path.exists() and path.read_text() != source:
        raise RuntimeError(f'Different experiment code already exists at {path}; start a fresh runtime.')
    path.write_text(source)
CLI = CODE / 'v5_experiment.py'
def cli(*args):
    subprocess.run([sys.executable, str(CLI), *map(str, args)], check=True)
cli('plan')


## 3. Upload the corrected dataset if it is not already saved

Select the complete corrected ZIP. You can upload the exclusion CSV alongside it. The next cell also lists likely exclusion-ledger files inside the ZIP; inspect them before mapping any nonstandard columns.


In [ ]:
import shutil, zipfile, json
from google.colab import files
ARCHIVE_NAME = 'YOLO_V5_CommandCenter_B187_QC_selected_merged1572.zip'
ARCHIVE = WORK / ARCHIVE_NAME
EXCLUSIONS = WORK / 'exclusions_28.csv'
if not ARCHIVE.is_file():
    uploaded = files.upload()
    del uploaded
    source = Path('/content') / ARCHIVE_NAME
    assert source.is_file(), f'Upload the exact corrected ZIP: {ARCHIVE_NAME}'
    shutil.copy2(source, ARCHIVE)
cli('audit', '--archive', ARCHIVE)
with zipfile.ZipFile(ARCHIVE) as z:
    ledger_candidates = [n for n in z.namelist() if n.lower().endswith('.csv')
                         and any(k in n.lower() for k in ('exclude', 'exclusion', 'uncert', 'decision'))]
    print('Possible QC/exclusion ledgers:', *ledger_candidates, sep='\n')


## 4. Provide the 28-image exclusion ledger

The ledger must contain an `image_path` column with exactly 28 unique paths matching the ZIP's manifest (for example `images/train/name.jpg`). If the ZIP includes that exact ledger, set `LEDGER_MEMBER` below to its listed path. Otherwise upload a mapped CSV. A JSON list of exact paths is also supported by the CLI. Do not guess identities from row numbers or exclude arbitrary photos. Preparation refuses a missing or invalid list.


In [ ]:
LEDGER_MEMBER = ''  # Set this only if the ZIP contains the exact, canonical 28-image CSV.
if not EXCLUSIONS.is_file():
    if LEDGER_MEMBER:
        with zipfile.ZipFile(ARCHIVE) as z:
            EXCLUSIONS.write_bytes(z.read(LEDGER_MEMBER))
    else:
        print('Upload the 28-image CSV with an image_path column.')
        uploaded = files.upload()
        csv_names = [n for n in uploaded if n.lower().endswith('.csv')]
        assert len(csv_names) == 1, 'Upload exactly one mapped exclusion CSV.'
        EXCLUSIONS.write_bytes(uploaded[csv_names[0]])
        del uploaded
print(EXCLUSIONS.read_text(encoding='utf-8-sig')[:5000])


## 5. Build the six-class dataset once

Image bytes stay unchanged; bottle rows are removed. Mixed images and bottle-only background negatives are retained. This creates one group-aware split used by every candidate. Previously held-out photos remain outside training. Reopening the notebook verifies the saved dataset instead of rebuilding a different split.


In [ ]:
if DATA.exists():
    cli('verify', '--dataset', DATA)
else:
    cli('prepare', '--archive', ARCHIVE, '--exclusions', EXCLUSIONS, '--output', DATA)


## 6. Train the baseline — 40 epochs

This starts real GPU training. It retrains the six-class model from general pretrained YOLO11s weights. It does not resume the old seven-class checkpoint. Use the epoch timings to estimate the remaining campaign. Standard best/last checkpoints are saved after epochs. If an interrupted run has a verified failure receipt, the campaign can resume it; an abrupt runtime kill may need manual recovery.


In [ ]:
# Run the first stage only. The campaign cell below skips this run once verified complete.
baseline = RUNS / 'baseline_seed57'
if (baseline / 'run.json').is_file():
    baseline_record = json.loads((baseline / 'run.json').read_text())
    if baseline_record.get('status') == 'complete':
        report = json.loads(subprocess.check_output([
            sys.executable, str(CLI), 'status', '--dataset', str(DATA), '--runs', str(RUNS)], text=True))
        assert {'trial': 'baseline', 'seed': 57} in report['completed_runs'], 'Baseline provenance is not valid.'
        print(json.dumps(report, indent=2))
    else:
        cli('train', '--dataset', DATA, '--output', baseline, '--trial', 'baseline', '--device', '0', '--resume')
else:
    cli('train', '--dataset', DATA, '--output', baseline, '--trial', 'baseline', '--device', '0')


## 7. Export results now or after more trials

Run this cell after the baseline and again whenever you want to return partial results. The ZIP includes dataset provenance and run reports, logs, curves, per-image matching stats and overlays. Model weights stay in the persistent run folders so the report remains manageable. No accuracy claim is made if a run failed or remains incomplete.


In [ ]:
EXPORT = Path('/content/V5_Six_Class_Unified_Results.zip')
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('dataset_receipt.json', 'data.yaml', 'train.txt', 'val.txt', 'test.txt'):
        p = DATA / name
        if p.is_file(): z.write(p, 'data/' + name)
    if RUNS.exists():
        for p in sorted(RUNS.rglob('*')):
            if p.is_file() and p.suffix.lower() in {'.json', '.csv', '.yaml', '.txt', '.png', '.jpg'}:
                z.write(p, 'runs/' + str(p.relative_to(RUNS)))
print('Report ZIP:', EXPORT, 'bytes:', EXPORT.stat().st_size)
files.download(str(EXPORT))


## 8. Run the six unified variants and repeat confirmations

Run this only when ready for the remaining GPU time. It runs U1 (gentler augmentation), U2 (1024 pixels), U3 (bounded weak-class balancing), U4 (lower learning rate), U5 (YOLO11m), and U6 (supported combination or 80-epoch convergence). Trials normally use 40 epochs. It then repeats the baseline and selected candidate at seeds 58 and 59. Valid completed runs are skipped, allowing the campaign to continue across sessions.

Afterward rerun the export cell. Return the ZIP for analysis of class support, repeat variability and paired group-bootstrap intervals. Specialist training is deliberately unavailable here and will follow only if the measured results justify it. Reserved test images are never used to pick a candidate.


In [ ]:
cli('campaign', '--dataset', DATA, '--runs', RUNS, '--device', '0')
cli('status', '--dataset', DATA, '--runs', RUNS)
